In [ ]:
# import packages
import pandas as pd
from bs4 import BeautifulSoup
import requests

In [ ]:
# assign the URL of the page to be crawled & request the page  
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'
page = requests.get(url)
soup = BeautifulSoup(page.text, 'html.parser')
print(soup)

In [ ]:
# find the first table on the page
table = soup.find_all('table', {'class':'wikitable'})[0]
print(table)

In [ ]:
# get the name of the columns in the table
for i in table.find_all('th'):
    print(i.text)

In [ ]:
# assign the column names to a list
titles = [header.text.strip() for header in table.find_all('th')]
print(titles)

In [ ]:
# iterate through all the rows and extract the data, clean and append to the DataFrame
rows = []
for i in table.find_all('tr')[1:]:
    row_data = []  
    for j in i.find_all('td'):
        row_data.append(j.text.strip())
    rows.append(row_data)
print(rows)

In [ ]:
# create a DataFrame from the list of rows
df = pd.DataFrame(rows, columns=titles)
df.head(10)

In [ ]:
# check the information of the DataFrame
df.info()

In [ ]:
# check the summary statistics of the DataFrame
df.describe()

In [ ]:
# split the Headquaters column into city and state for better representation
df.iloc[:, :] = df.iloc[:, :].astype({'Rank': int, 'Name': str, 'Industry': str, 'Revenue (USD millions)': str, 'Revenue growth': str, 'Employees': str, 'Headquarters': str})
df['Revenue (USD millions)'] = df['Revenue (USD millions)'].str.replace(',', '').astype(float).div(1000).round(2)     
df['Revenue growth'] = df['Revenue growth'].str.replace('%', '').astype(float)  
df['Employees'] = df['Employees'].str.replace(',', '').astype(int)


In [ ]:
# clean the new columns
df[['city', 'state']] = df['Headquarters'].str.rsplit(',',n= 1, expand=True)  
df['state'] = df['state'].str.strip()
df['city'] = df['city'].str.strip()

In [ ]:
# make the column names lower case and replace spaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
# rename the revenue column
df.rename(columns={"revenue_(usd_millions)" : "revenue(usd_billions)", "revenue_growth" : "revenue_growth(%)"}, inplace=True)   

In [ ]:
# drop the original headquarters column
df.drop('headquarters', axis=1, inplace=True)

In [ ]:
# save the DataFrame to a CSV file for visualization
df.to_csv(r"C:\Users\joysn\Desktop\biggest_companies.csv", index=False)